<a href="https://colab.research.google.com/github/MeenakshiRajpurohit/CMPE-297-Natural-Language-Processing/blob/main/CMPE259_Week6_RAG_Homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <span style="color:blue">CMPE 259 — Week 6 Homework: Retrieval-Augmented Generation (RAG) with a Vector Knowledge Base</span>

**Name:** `Meenakshi Rajpurohit`  **Course:** CMPE 259 Natural Language Processing, Fall 2026  **Topic:** Knowledge Integration, RAG, and Grounding Models with Data

## Objective

Build my own RAG system: upsert a PDF knowledge base into a Pinecone vector database, retrieve the most relevant chunks for each question with OpenAI embeddings, and pass them to a GPT model so its answers are grounded in that knowledge.

### My knowledge base

**`Larkspur_Wren_S2_Knowledge_Base.pdf`** — a 7-page product and customer-support knowledge base for the *Wren S2 robot vacuum & mop* made by *Larkspur Home Robotics*. It covers specifications, setup, cleaning modes, maintenance, error codes, warranty, returns, shipping, the Care+ protection plans, privacy, accessories and support contacts.

The company and product are **fictional on purpose**. A GPT model cannot know these facts from its pre-training, so every correct answer has to come from retrieval. That makes the effect of RAG easy to see and measure.

### What this notebook does

| Task | Section |
|---|---|
| Information retrieval: a customer-support chatbot with memory ("WrenHelper") | 6 |
| Augmented generation: the same questions answered **without** vs **with** the knowledge base | 7 |
| Fact-checking: claims verified against retrieved evidence, with confidence from **logprobs** | 8 |
| Guardrails: out-of-scope questions should get "I'm not certain…" | 9 |
| Evaluation: retrieval hit rate and answer accuracy on a small test set | 10 |

### Pipeline

```
INDEXING:  PDF ─► extract & clean (PyMuPDF) ─► overlapping line chunks ─► OpenAI embeddings (1536-d) ─► Pinecone (namespace)
QUERYING:  question ─► embed ─► Pinecone top-k search ─► context wrapped in #### ─► GPT ─► grounded answer
```

## Before you run: fill in 3 things

1. **OpenAI API key** – section 0.2
2. **Pinecone API key** – section 0.2 (free key at https://www.pinecone.io/)
3. **PDF path** – section 0.4 (upload `Larkspur_Wren_S2_Knowledge_Base.pdf` to Colab first)

Then choose **Runtime → Run all**. The last code cell (section 11) waits for you to type questions; type `exit` to finish.

# 0. Setup
## 0.1 Install and import libraries

In [1]:
!pip install -q -U openai pinecone pymupdf

In [2]:
import os
import re
import math
import time
import textwrap

import numpy as np
import pandas as pd
from IPython.display import display, HTML

import openai
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
import pymupdf  # PyMuPDF (the class demo imported it as `fitz`, which is now deprecated)
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# ==================== 🔑 PASTE YOUR API KEYS HERE ====================
OPENAI_API_KEY   = "PASTE_YOUR_OPENAI_API_KEY_HERE"
PINECONE_API_KEY = "PASTE_YOUR_PINECONE_API_KEY_HERE"
# =====================================================================


def _from_colab_secrets(name):
    """Read a key from Colab Secrets if it exists there (returns None otherwise)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


if OPENAI_API_KEY.startswith("PASTE_"):
    OPENAI_API_KEY = _from_colab_secrets("OPENAI_API_KEY") or OPENAI_API_KEY
if PINECONE_API_KEY.startswith("PASTE_"):
    PINECONE_API_KEY = _from_colab_secrets("PINECONE_API_KEY") or PINECONE_API_KEY

assert not OPENAI_API_KEY.startswith("PASTE_"), "Please paste your OpenAI API key in the cell above."
assert not PINECONE_API_KEY.startswith("PASTE_"), "Please paste your Pinecone API key in the cell above."

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

client = OpenAI()
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
print("OpenAI and Pinecone clients are ready.")

OpenAI and Pinecone clients are ready.


## 0.3 Model helpers (same as the class notebook)

`response_request` calls the OpenAI **Responses API** for chat answers. `get_completion` calls the Chat Completions API and is used in section 8 because it can return **logprobs**.

In [4]:
MODEL = "gpt-4o-mini"


def response_request(
    messages,
    model=MODEL,
    tools=None,
    tool_choice="auto",
    conversation=None,
    max_output_tokens=None,
    temperature=None,   # will be ignored for gpt-5 models
):
    if str(model).startswith("gpt-5"):
        # Do NOT send temperature for gpt-5 family
        return client.responses.create(
            model=model,
            input=messages,
            tools=tools,
            tool_choice=tool_choice,
            conversation=conversation,
            max_output_tokens=max_output_tokens,
        )
    else:
        # Non–gpt-5 models can receive temperature
        return client.responses.create(
            model=model,
            input=messages,
            tools=tools,
            tool_choice=tool_choice,
            conversation=conversation,
            max_output_tokens=max_output_tokens,
            temperature=temperature,
        )


def get_completion(
    messages,
    model="gpt-4o-mini",
    max_tokens=500,
    temperature=0,
    stop=None,
    seed=123,
    tools=None,
    logprobs=None,      # if True, return the log probability of each output token
    top_logprobs=None,  # how many alternative tokens to return for each position
):
    params = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stop": stop,
        "seed": seed,
        "logprobs": logprobs,
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools
    return client.chat.completions.create(**params)

## 0.4 Knowledge base file

Upload **`Larkspur_Wren_S2_Knowledge_Base.pdf`** to Colab: open the 📁 **Files** panel on the left, click **Upload**, and pick the file. It will appear at `/content/Larkspur_Wren_S2_Knowledge_Base.pdf`. If you put it somewhere else, change the path below.

In [5]:
# ==================== 📄 KNOWLEDGE BASE FILE PATH ====================
filename = "/content/Larkspur_Wren_S2_Knowledge_Base.pdf"
# =====================================================================

# Optional: choose the file with an upload button instead (uncomment the 3 lines below)
# from google.colab import files
# uploaded = files.upload()
# filename = "/content/" + next(iter(uploaded))

# Optional: read it from Google Drive instead (uncomment, then fix the folder name)
# from google.colab import drive
# drive.mount("/content/drive")
# filename = "/content/drive/MyDrive/<your folder>/Larkspur_Wren_S2_Knowledge_Base.pdf"

assert os.path.exists(filename), f"PDF not found at {filename}. Upload it or fix the path above."
print("Knowledge base file name:", filename)

Knowledge base file name: /content/Larkspur_Wren_S2_Knowledge_Base.pdf


# 1. Embeddings and semantic similarity

Retrieval works because sentences with the same meaning get nearby embedding vectors, even when they share few words. Below, sentences 1 and 2 ask the same thing in different words; sentence 3 is about something else.

In [6]:
embed_model = "text-embedding-3-small"   # produces 1536-dimensional vectors

sentences = [
    "How long does the robot's battery last on a single charge?",
    "What is the runtime of the vacuum before it needs recharging?",
    "How do I send the vacuum back for a refund?",
]

res = client.embeddings.create(input=sentences, model=embed_model)
vectors = [np.array(d.embedding) for d in res.data]

print("Embedding length:", len(vectors[0]))
print("First 5 values of sentence 1:", np.round(vectors[0][:5], 4))

Embedding length: 1536
First 5 values of sentence 1: [ 0.0489 -0.0234  0.0518 -0.0206 -0.0302]


In [7]:
def cosine_similarity_vectors(vector1, vector2):
    return cosine_similarity([vector1], [vector2])[0][0]


print("Sentence 1 vs 2 (same meaning, different words):", round(cosine_similarity_vectors(vectors[0], vectors[1]), 4))
print("Sentence 1 vs 3 (different topic):             ", round(cosine_similarity_vectors(vectors[0], vectors[2]), 4))

Sentence 1 vs 2 (same meaning, different words): 0.4897
Sentence 1 vs 3 (different topic):              0.1674


# 2. Connect to the vector database (create the index if it does not exist)

Index names may only use lowercase letters, numbers and hyphens. The free Pinecone plan limits how many indexes you can have; if you hit that limit, set `index_name = "sjsunlp"` to reuse the class index (it has the same 1536 dimensions). This homework uses its own **namespace**, so it will not mix with the class data.

In [8]:
index_name = "cmpe259-hw6-rag"   # your own vector database index name
NAMESPACE  = "wren_s2_kb"        # every chunk of this knowledge base goes into this namespace
EMBED_DIM  = 1536                # must match text-embedding-3-small

# exists-or-create
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        deletion_protection="disabled",
        tags={"environment": "development"},
        vector_type="dense",
    )
    print(f"Created index '{index_name}'.")
else:
    print(f"Index '{index_name}' already exists, reusing it.")

# wait until ready
while not pc.describe_index(index_name).status["ready"]:
    time.sleep(1)

assert pc.describe_index(index_name).dimension == EMBED_DIM, "This index has a different dimension; choose another index_name."
print("Index ready.")

# connect to index and view its stats
index = pc.Index(index_name)
index.describe_index_stats()

Created index 'cmpe259-hw6-rag'.
Index ready.


DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='cosine', namespaces=0)

# 3. Extract and clean the PDF text

Same cleaning function as the class demo: fix hyphenated line breaks, unwrap soft line breaks, then re-wrap the text into lines of at most 120 characters. Each line becomes the unit for chunking.

In [9]:
def extract_clean_120_list(pdf_path: str) -> list[str]:
    """Return cleaned PDF text as a list of lines (wrapped at 120 chars)."""
    doc = pymupdf.open(pdf_path)
    raw = "\n".join(p.get_text() for p in doc)

    # Fix hyphenations and unwrap soft breaks
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', raw)
    text = re.sub(r'(?<![.!?;:])\n(?!\n)', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text).strip()
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Split into paragraphs, wrap to 120 chars
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    wrapped_paras = [
        textwrap.fill(p, width=120, break_long_words=False, break_on_hyphens=False)
        for p in paras
    ]

    # Finally, flatten paragraphs into list of lines
    lines = []
    for para in wrapped_paras:
        lines.extend(para.splitlines())
        lines.append("")  # keep blank line between paragraphs
    if lines and lines[-1] == "":
        lines.pop()  # remove trailing blank line

    return lines

In [10]:
doc = extract_clean_120_list(filename)
print("Total Length: ", len(doc))
print("First 3 lines:")
for line in doc[:3]:
    print("  ", line)

Total Length:  132
First 3 lines:
   Page 1 LARKSPUR HOME ROBOTICS Wren S2 Robot Vacuum & Mop Product and Customer Support Knowledge Base Version 3.2 ·
   Effective September 1, 2026 · For customers and support agents About this document. This knowledge base describes the
   Wren S2 robot vacuum and mop made by Larkspur Home Robotics, Inc. It covers specifications, setup, cleaning modes,


# 4. Chunk the document and upsert it to the vector database

`nlp_upsert` keeps the class signature `nlp_upsert(filename, index_name, name_space, nlp_id, chunk_size, stride)`, where `chunk_size` is the number of lines per chunk and `stride` is the number of lines shared by neighboring chunks.

**Small improvements over the class demo**

1. **True sliding window.** The class loop moves forward by `chunk_size` and then steps back by `stride`, so only the first two chunks overlap and the last `stride` lines of the file are never embedded (in the class output, the end of the Week 10 schedule and the Week 11 row are missing). Here the window moves forward by `chunk_size − stride`, so every pair of neighbors overlaps and the final lines are always included.
2. **Lines are joined with a line break** instead of an empty string, so words at line ends no longer merge (the class output shows "broadened**the**").
3. **One batched embedding call** instead of one call per chunk, with a limited retry instead of an endless loop.
4. **Richer metadata** on every vector: source file, chunk number and line range, which makes retrieved chunks easy to trace.

In [11]:
def build_chunks(lines, chunk_size, stride):
    """Split a list of lines into overlapping chunks of `chunk_size` lines, sharing `stride` lines."""
    assert 0 <= stride < chunk_size, "stride (overlap) must be smaller than chunk_size"
    step = chunk_size - stride
    chunks = []
    for start in range(0, len(lines), step):
        end = min(len(lines), start + chunk_size)
        text = "\n".join(lines[start:end]).strip()
        if text:
            chunks.append({"text": text, "line_start": start, "line_end": end - 1})
        if end == len(lines):
            break
    return chunks


def embed_texts(texts, batch_size=64, max_retries=5):
    """Embed a list of texts with OpenAI, in batches, retrying temporary errors."""
    all_vectors = []
    for b in range(0, len(texts), batch_size):
        batch = texts[b:b + batch_size]
        for attempt in range(max_retries):
            try:
                res = client.embeddings.create(input=batch, model=embed_model)
                all_vectors.extend(d.embedding for d in res.data)
                break
            except openai.AuthenticationError:
                raise  # a wrong key will not fix itself, so stop right away
            except Exception as e:
                if attempt == max_retries - 1:
                    raise
                wait = 2 ** attempt
                print(f"  embedding call failed ({type(e).__name__}); retrying in {wait}s ...")
                time.sleep(wait)
    return all_vectors


def nlp_upsert(filename, index_name, name_space, nlp_id, chunk_size, stride, verbose=True):
    """
    Upsert a whole PDF file to the Pinecone vector database.

    Parameters:
    filename (str): The PDF file name (path).
    index_name (str): The Pinecone index name.
    name_space (str): The namespace for all chunks of this document.
    nlp_id (str): A common ID prefix for the document's vectors.
    chunk_size (int): How many lines make one chunk.
    stride (int): How many lines neighboring chunks share (overlap).

    Returns:
    int: Number of vectors upserted.
    """
    print("extracting pdf...")
    doc = extract_clean_120_list(filename)
    print("extracting finished! Total Lines: ", len(doc))

    chunks = build_chunks(doc, chunk_size, stride)
    print(f"Built {len(chunks)} chunks (chunk_size={chunk_size} lines, overlap={stride} lines)")

    # Create embeddings of all chunk texts
    embeds = embed_texts([c["text"] for c in chunks])
    print("Embeds length:", len(embeds[0]))

    source = os.path.basename(filename)
    vectors = []
    for n, (chunk, embed) in enumerate(zip(chunks, embeds), start=1):
        metadata = {
            "text": chunk["text"],
            "source": source,
            "chunk": n,
            "line_start": chunk["line_start"],
            "line_end": chunk["line_end"],
        }
        vectors.append({"id": f"{nlp_id}_{n}", "values": embed, "metadata": metadata})
        if verbose:
            print("-" * 80)
            print(f"Chunk {n} (lines {chunk['line_start']}-{chunk['line_end']}):")
            print(chunk["text"])

    # upsert to Pinecone, into the chosen namespace
    index = pc.Index(index_name)
    for b in range(0, len(vectors), 100):
        index.upsert(vectors=vectors[b:b + 100], namespace=name_space)

    print("=" * 80)
    print("Upserted vector count is: ", len(vectors))
    return len(vectors)

### <span style="color:#00AA00">Building the Knowledge Base</span>

The namespace is cleared first, so re-running this cell never leaves old chunks behind. To add more PDFs to the same knowledge base, call `nlp_upsert` again with a different file and a different `nlp_id` (and skip the clearing step).

In [12]:
def clear_namespace(name_space):
    """Delete all vectors in a namespace (Pinecone raises an error if it does not exist yet)."""
    try:
        index.delete(delete_all=True, namespace=name_space)
        print(f"Cleared namespace '{name_space}'.")
    except Exception:
        print(f"Namespace '{name_space}' does not exist yet, nothing to clear.")


CHUNK_SIZE = 6   # lines per chunk (each line is up to 120 characters)
STRIDE     = 2   # lines of overlap between neighboring chunks

clear_namespace(NAMESPACE)
n_upserted = nlp_upsert(filename, index_name, NAMESPACE, "wren", CHUNK_SIZE, STRIDE)
# signature: nlp_upsert(filename, index_name, name_space, nlp_id, chunk_size, stride)

# Example for a second file:
# nlp_upsert("/content/another_file.pdf", index_name, NAMESPACE, "doc2", CHUNK_SIZE, STRIDE)

Namespace 'wren_s2_kb' does not exist yet, nothing to clear.
extracting pdf...
extracting finished! Total Lines:  132
Built 33 chunks (chunk_size=6 lines, overlap=2 lines)
Embeds length: 1536
--------------------------------------------------------------------------------
Chunk 1 (lines 0-5):
Page 1 LARKSPUR HOME ROBOTICS Wren S2 Robot Vacuum & Mop Product and Customer Support Knowledge Base Version 3.2 ·
Effective September 1, 2026 · For customers and support agents About this document. This knowledge base describes the
Wren S2 robot vacuum and mop made by Larkspur Home Robotics, Inc. It covers specifications, setup, cleaning modes,
maintenance, error codes, warranty, returns, shipping, the Larkspur Care+ protection plans, privacy, accessories and
support contacts. Larkspur Home Robotics and the Wren product line are fictional and were created for the CMPE 259 Week
6 Retrieval-Augmented Generation (RAG) assignment; all phone numbers and web addresses are placeholders. 1. Company
-----

**Check that the build succeeded.** Serverless indexes take a few seconds before new vectors become searchable, so this cell waits until the namespace shows all of them.

In [13]:
def namespace_vector_count(name_space):
    stats = index.describe_index_stats()
    try:
        return int(stats.namespaces[name_space].vector_count)
    except Exception:
        try:
            return int(stats["namespaces"][name_space]["vector_count"])
        except Exception:
            return 0


for _ in range(30):
    if namespace_vector_count(NAMESPACE) >= n_upserted:
        break
    time.sleep(2)

print(f"Vectors in namespace '{NAMESPACE}':", namespace_vector_count(NAMESPACE))
index.describe_index_stats()

Vectors in namespace 'wren_s2_kb': 33


DescribeIndexStatsResponse(dimension=1536, total_vector_count=33, metric='cosine', namespaces=1)

# 5. Test retrieval on its own

Before adding the language model, check that semantic search returns the right chunks. A higher cosine score means a closer match.

In [14]:
TOP_K = 4   # number of chunks retrieved per question


def search_kb(query, name_space=NAMESPACE, top_k=TOP_K):
    """Embed the query and return the top-k matching chunks from Pinecone."""
    xq = client.embeddings.create(input=[query], model=embed_model).data[0].embedding
    res = index.query(vector=xq, top_k=top_k, include_metadata=True, namespace=name_space)
    return res.matches


def show_matches(query, top_k=TOP_K):
    print(f"Query: {query}\n")
    for rank, m in enumerate(search_kb(query, top_k=top_k), start=1):
        print(f"#{rank}  id={m.id}  score={m.score:.4f}")
        preview = m.metadata["text"].replace("\n", " ")
        print(textwrap.indent(textwrap.fill(preview, 110), "    "))
        print()


show_matches("What does error code E9 mean?")

Query: What does error code E9 mean?

#1  id=wren_17  score=0.5045
    sure nothing is pressing on the LiDAR turret, remove debris around it, and check that the top cover is closed.
    If E9 returns after a restart, contact Larkspur Support. E10 Wi-Fi connection failed Confirm the network is
    2.4 GHz, move the router closer or restart it, then reconnect in the app. E11 Dock bag full or missing Replace
    the dock dust bag. E12 Robot trapped Move the robot to an open area. Add a No-Go Zone around the spot where it
    got stuck. E15 Battery temperature out of range Move the robot to a room between 0 and 40 °C and wait 30
    minutes before charging. E21 Suction fan overload Clean the HEPA filter and dustbin. If E21 repeats, contact
    Larkspur Support. Restart and factory reset To

#2  id=wren_15  score=0.4803
    the error code, and the code appears in the Larkspur Home app. Code Meaning What to do E1 Wheel stuck or
    lifted Check both drive wheels for hair or debris and place 

In [15]:
show_matches("Can I get my money back if I don't like the robot?")

Query: Can I get my money back if I don't like the robot?

#1  id=wren_25  score=0.4208
    Page 6 • Return shipping is free; Larkspur emails a prepaid return label. Original packaging is preferred but
    not required. • Returns missing accessories (dock, brushes, mop pads, power cord) are charged a 15% restocking
    fee. Complete returns have no restocking fee. • Returned items are inspected within 3 business days of
    arrival, and refunds reach the original payment method 5 to 7 business days after inspection. • Products
    bought from other retailers follow that retailer's return policy. • Returning a robot does not cancel a
    Larkspur Care+ plan; cancel the plan separately in the app. Shipping • Larkspur ships to the United States and
    Canada only. • United States: free standard shipping (3 to 5

#2  id=wren_11  score=0.3874
    robot. • Edge Clean: follows walls and furniture edges with the side brush for a deeper perimeter clean. •
    Quiet Hours: default 10:00 p.m. to

# 6. RAG chatbot: "WrenHelper"
## 6.1 System prompt and chat memory

As in the class demo, retrieved knowledge is wrapped in the `####` delimiter and the model is told to answer only from it. `chatContext` holds the conversation memory.

In [16]:
delimiter = "####"

SYSTEM_PROMPT = f"""\
You are WrenHelper, a friendly and precise customer-support assistant for Larkspur Home Robotics, \
focused on the Wren S2 robot vacuum and mop.

Answer the customer's question using ONLY the knowledge-base excerpts in the context marked by {delimiter}. \
Quote numbers, prices, time limits and error codes exactly as they appear in the context. \
For troubleshooting, give short numbered steps. Keep answers concise (about 150 words or fewer).

If the context does not contain the information needed, or the question is unrelated to Larkspur products, respond with: \
"I'm not certain, as the knowledge base doesn't contain the necessary information." \
and suggest contacting Larkspur Customer Support. Never invent facts that are not in the context.
"""

chatContext = [{"role": "system", "content": SYSTEM_PROMPT}]
chatContext

[{'role': 'system',
  'content': 'You are WrenHelper, a friendly and precise customer-support assistant for Larkspur Home Robotics, focused on the Wren S2 robot vacuum and mop.\n\nAnswer the customer\'s question using ONLY the knowledge-base excerpts in the context marked by ####. Quote numbers, prices, time limits and error codes exactly as they appear in the context. For troubleshooting, give short numbered steps. Keep answers concise (about 150 words or fewer).\n\nIf the context does not contain the information needed, or the question is unrelated to Larkspur products, respond with: "I\'m not certain, as the knowledge base doesn\'t contain the necessary information." and suggest contacting Larkspur Customer Support. Never invent facts that are not in the context.\n'}]

## 6.2 Retrieval function (formal running)

Same idea as the class `retrieve()`: fetch the top-k chunks, keep adding them while they fit in a token budget, and wrap the result in the delimiter.

In [17]:
# Rough token estimate: tokens ≈ words / 0.75
WORD_TOKEN_RATIO = 0.75


def approx_tokens(text: str) -> int:
    words = len(re.findall(r"\S+", text))
    return math.ceil(words / WORD_TOKEN_RATIO)


limit_tokens = 8000  # token budget for the retrieved context block


def retrieve(query, name_space=NAMESPACE, top_k=TOP_K, followup_query=None):
    matches = search_kb(query, name_space, top_k)
    if followup_query:  # used by rag_chat: add up to 2 more chunks found with the previous question included
        seen = {m.id for m in matches}
        matches += [m for m in search_kb(followup_query, name_space, 2) if m.id not in seen]
    contexts = [m.metadata["text"] for m in matches]

    # Build context block
    context = ""
    used_tokens = approx_tokens(delimiter + delimiter)  # account for wrappers
    for c in contexts:
        c_tokens = approx_tokens(c)
        if used_tokens + c_tokens > limit_tokens:
            break
        context += c + "\n\n"
        used_tokens += c_tokens

    # Wrap with delimiters
    return delimiter + "\n" + context.strip() + "\n" + delimiter

## 6.3 Step-by-step walkthrough (same flow as the class demo)
## <span style="color:#00AA00">Question Put Here</span>

In [18]:
query = "My Wren S2 keeps showing error E9. What should I do?"
# query = input("User Input: ")   # uncomment to type your own question, as in the class demo

query_message = {"role": "user", "content": f"\n{query}\n"}
print("query message:", query_message)

query message: {'role': 'user', 'content': '\nMy Wren S2 keeps showing error E9. What should I do?\n'}


In [19]:
# first we retrieve relevant chunks from Pinecone
retrieved_knowledge = retrieve(query, NAMESPACE)
print(retrieved_knowledge)

####
sure nothing is pressing on the LiDAR turret, remove debris around it, and check that the top cover is closed. If E9
returns after a restart, contact Larkspur Support. E10 Wi-Fi connection failed Confirm the network is 2.4 GHz, move the
router closer or restart it, then reconnect in the app. E11 Dock bag full or missing Replace the dock dust bag. E12
Robot trapped Move the robot to an open area. Add a No-Go Zone around the spot where it got stuck. E15 Battery
temperature out of range Move the robot to a room between 0 and 40 °C and wait 30 minutes before charging. E21 Suction
fan overload Clean the HEPA filter and dustbin. If E21 repeats, contact Larkspur Support. Restart and factory reset To

the contacts on the robot and dock with a dry cloth once a month. Not applicable Water tank: empty the water tank after
each use. Use only plain tap water or Larkspur FloorCare solution (10 mL of FloorCare per full tank). Never add bleach,
vinegar, essential oils or other cleaning products; 

In [20]:
# infuse the knowledge into the messages sent to the model
knowledge_message = {"role": "system", "content": f"\n{retrieved_knowledge}\n"}
context_query_knowledge = chatContext + [knowledge_message, query_message]

print("Messages sent to the model:")
for m in context_query_knowledge:
    print(f"  {m['role']:>9}: {m['content'][:90]!r}...")

Messages sent to the model:
     system: 'You are WrenHelper, a friendly and precise customer-support assistant for Larkspur Home Ro'...
     system: '\n####\nsure nothing is pressing on the LiDAR turret, remove debris around it, and check tha'...
       user: '\nMy Wren S2 keeps showing error E9. What should I do?\n'...


## <span style="color:#CC00CC">ChatBot Response Here</span>

In [21]:
response = response_request(context_query_knowledge, temperature=0)
print(response.output_text)

To troubleshoot error E9 on your Wren S2, follow these steps:

1. Ensure nothing is pressing on the LiDAR turret.
2. Remove any debris around the LiDAR turret.
3. Check that the top cover is closed.

If error E9 returns after a restart, please contact Larkspur Support for further assistance.


In [22]:
# save this turn in the chat memory (the retrieved knowledge itself is not stored)
chatContext.append(query_message)
chatContext.append({"role": "assistant", "content": response.output_text})
print("Messages in chat memory:", len(chatContext))

Messages in chat memory: 3


## 6.4 The same steps wrapped in functions

* `rag_answer` runs one retrieval + generation step.
* `rag_chat` adds memory. A follow-up such as *"What does it include?"* has almost no keywords of its own, so `rag_chat` also searches with the previous question attached and adds up to 2 extra chunks from that search.

In [23]:
def rag_answer(query, history=None, followup_query=None, name_space=NAMESPACE, top_k=TOP_K):
    """One RAG turn. Returns (answer, retrieved_knowledge)."""
    if history is None:
        history = [{"role": "system", "content": SYSTEM_PROMPT}]
    retrieved_knowledge = retrieve(query, name_space, top_k, followup_query=followup_query)
    knowledge_message = {"role": "system", "content": f"\n{retrieved_knowledge}\n"}
    query_message = {"role": "user", "content": f"\n{query}\n"}
    response = response_request(history + [knowledge_message, query_message], temperature=0)
    return response.output_text, retrieved_knowledge


def rag_chat(query, show_context=False):
    """Chat with memory: answers the question and stores the turn in chatContext."""
    previous_questions = [m["content"].strip() for m in chatContext if m["role"] == "user"]
    followup_query = (previous_questions[-1] + " " + query) if previous_questions else None

    answer, retrieved_knowledge = rag_answer(query, history=chatContext, followup_query=followup_query)
    if show_context:
        print(retrieved_knowledge, "\n")

    chatContext.append({"role": "user", "content": f"\n{query}\n"})
    chatContext.append({"role": "assistant", "content": answer})
    return answer


def wrap(text, width=110, indent="    "):
    return "\n".join(textwrap.fill(p, width, initial_indent=indent, subsequent_indent=indent) if p.strip() else ""
                     for p in str(text).split("\n"))

In [24]:
# A short multi-turn conversation: the second question depends on the first
for q in ["How much does the Larkspur Care+ Premium plan cost?",
          "What does it include that Care+ Basic does not?"]:
    print("User:", q)
    print("WrenHelper:")
    print(wrap(rag_chat(q)))
    print("-" * 80)

User: How much does the Larkspur Care+ Premium plan cost?
WrenHelper:
    The Larkspur Care+ Premium plan costs $9.99 per month or $99 per year.
--------------------------------------------------------------------------------
User: What does it include that Care+ Basic does not?
WrenHelper:
    The Larkspur Care+ Premium plan includes features that Care+ Basic does not, such as:

    1. Extended warranty coverage while subscribed, up to 4 years from the original purchase date.
    2. Accidental damage protection, covering drops and spills with 1 claim per 12 months (with a $49 service
    fee per claim).
    3. Advance replacement, where a replacement ships before you send back the defective unit.
    4. A Consumables Kit, which includes free supplies every 6 months: 2 side brushes, 1 HEPA filter, and 2
    mop pads.
    5. Priority phone support with an average wait under 5 minutes.

    Care+ Basic does not offer these additional benefits.
--------------------------------------------

# 7. Experiment 1 – Without RAG vs. with RAG

The same questions go to the same model twice: once with no knowledge base, once through the RAG pipeline. Because the product is fictional, the model without retrieval has no way to know the answers.

In [25]:
def ask_without_rag(question):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer the user's question."},
        {"role": "user", "content": question},
    ]
    return response_request(messages, temperature=0).output_text


comparison_questions = [
    "What does error code E9 mean on the Larkspur Wren S2 robot vacuum?",
    "How long is the battery warranty on the Larkspur Wren S2?",
    "How much does Larkspur Care+ Premium cost per year?",
    "Can the Larkspur Wren S2 connect to a 5 GHz Wi-Fi network?",
]

for q in comparison_questions:
    no_rag = ask_without_rag(q)
    with_rag, _ = rag_answer(q)
    print("QUESTION:", q)
    print("\n  ✗ WITHOUT RAG:")
    print(wrap(no_rag))
    print("\n  ✓ WITH RAG:")
    print(wrap(with_rag))
    print("=" * 110)

QUESTION: What does error code E9 mean on the Larkspur Wren S2 robot vacuum?

  ✗ WITHOUT RAG:
    Error code E9 on the Larkspur Wren S2 robot vacuum typically indicates a problem with the vacuum's brush
    system, often related to a blockage or jam. Here are some steps you can take to troubleshoot and resolve
    the issue:

    1. **Check the Brushes**: Inspect the main brush and side brushes for any hair, debris, or obstructions.
    Remove any tangled hair or debris.

    2. **Clean the Brush Housing**: Ensure that the brush housing is clean and free from any blockages.

    3. **Inspect the Wheels**: Check the wheels for any obstructions that might prevent them from moving
    freely.

    4. **Reset the Vacuum**: Sometimes, simply turning the vacuum off and then back on can reset the error.

    5. **Consult the Manual**: If the issue persists, refer to the user manual for specific troubleshooting
    steps related to error code E9.

    If none of these steps resolve the issue,

# 8. Experiment 2 – Fact-checking with retrieved evidence and logprobs

For each claim, the pipeline retrieves evidence from the knowledge base and asks the model for a one-word verdict: **True**, **False** or **Unknown**. As in class section 1.1, `logprobs=True` shows how confident the model is in that verdict token. A second call writes a one-sentence justification.

In [26]:
FACT_CHECK_PROMPT = """You are a fact-checker for Larkspur Home Robotics.
Decide whether the CLAIM is supported by the EVIDENCE taken from the Wren S2 knowledge base.
Return only one word, and nothing else:
True    - the evidence clearly supports the claim
False   - the evidence clearly contradicts the claim
Unknown - the evidence does not contain enough information to decide

EVIDENCE:
{evidence}

CLAIM: {claim}"""

JUSTIFY_PROMPT = """The fact-check verdict for this claim is: {verdict}.
Using only the EVIDENCE, explain this verdict in one sentence.

EVIDENCE:
{evidence}

CLAIM: {claim}"""


def fact_check(claim, top_k=TOP_K):
    evidence = retrieve(claim, NAMESPACE, top_k)
    api_response = get_completion(
        [{"role": "user", "content": FACT_CHECK_PROMPT.format(evidence=evidence, claim=claim)}],
        max_tokens=3,
        logprobs=True,
        top_logprobs=3,
    )
    content = (api_response.choices[0].message.content or "").strip()
    verdict = content.split()[0].strip(".,") if content else "Unknown"
    top_logprobs = api_response.choices[0].logprobs.content[0].top_logprobs
    confidence = float(np.exp(top_logprobs[0].logprob))

    justification = response_request(
        [{"role": "user", "content": JUSTIFY_PROMPT.format(verdict=verdict, evidence=evidence, claim=claim)}],
        temperature=0,
    ).output_text
    return verdict, confidence, top_logprobs, justification

In [27]:
claims = [
    # (claim, expected verdict according to the knowledge base)
    ("The Wren S2 robot comes with a 2-year limited warranty.", "True"),
    ("The Wren S2 uses a camera to recognize obstacles.", "False"),
    ("The Wren S2 can connect to 5 GHz Wi-Fi networks.", "False"),
    ("Larkspur Care+ Premium includes accidental damage protection.", "True"),
    ("A Wren S2 delivered on December 15 can be returned until January 31 of the following year.", "True"),
    ("The Wren S2 runs for up to 4 hours in Eco mode.", "False"),
    ("The Wren S2 is manufactured in Vietnam.", "Unknown"),
]

fact_rows = []
for claim, expected in claims:
    verdict, confidence, top_logprobs, justification = fact_check(claim)

    html_content = f"<b>Claim:</b> {claim}<br>"
    for i, lp in enumerate(top_logprobs, start=1):
        html_content += (
            f"<span style='color: teal'>Output token {i}:</span> {lp.token}, "
            f"<span style='color: darkorange'>logprobs:</span> {lp.logprob:.4f}, "
            f"<span style='color: purple'>linear probability:</span> {np.round(np.exp(lp.logprob) * 100, 2)}%<br>"
        )
    html_content += f"<b>Why:</b> {justification}<br><br>"
    display(HTML(html_content))

    fact_rows.append({
        "claim": claim,
        "expected": expected,
        "model verdict": verdict,
        "confidence %": round(confidence * 100, 2),
        "correct": "✓" if verdict.lower() == expected.lower() else "✗",
    })

fact_df = pd.DataFrame(fact_rows)
display(fact_df)
print(f"Fact-checking accuracy: {(fact_df['correct'] == '✓').mean():.0%} ({(fact_df['correct'] == '✓').sum()}/{len(fact_df)})")

,claim,expected,model verdict,confidence %,correct
0,The Wren S2 robot comes with a 2-year limited ...,True,True,100.00,✓
1,The Wren S2 uses a camera to recognize obstacles.,False,False,100.00,✓
2,The Wren S2 can connect to 5 GHz Wi-Fi networks.,False,False,100.00,✓
3,Larkspur Care+ Premium includes accidental dam...,True,True,98.20,✓
4,A Wren S2 delivered on December 15 can be retu...,True,Unknown,62.22,✗
5,The Wren S2 runs for up to 4 hours in Eco mode.,False,False,100.00,✓
6,The Wren S2 is manufactured in Vietnam.,Unknown,Unknown,100.00,✓


Fact-checking accuracy: 86% (6/7)


# 9. Experiment 3 – Out-of-scope questions (guardrail)

When the knowledge base has no answer, the assistant should say *"I'm not certain…"* instead of making something up.

In [28]:
out_of_scope = [
    "What is the return policy for Dyson vacuums?",
    "Does the Wren S2 come in a red color?",
    "Who won the 2022 FIFA World Cup?",
]

for q in out_of_scope:
    answer, _ = rag_answer(q)
    print("QUESTION:", q)
    print(wrap(answer))
    print("-" * 80)

QUESTION: What is the return policy for Dyson vacuums?
    I'm not certain, as the knowledge base doesn't contain the necessary information. I suggest contacting
    Larkspur Customer Support.
--------------------------------------------------------------------------------
QUESTION: Does the Wren S2 come in a red color?
    I'm not certain, as the knowledge base doesn't contain the necessary information. Please contact Larkspur
    Customer Support for assistance.
--------------------------------------------------------------------------------
QUESTION: Who won the 2022 FIFA World Cup?
    I'm not certain, as the knowledge base doesn't contain the necessary information. Please contact Larkspur
    Customer Support for assistance.
--------------------------------------------------------------------------------


# 10. Experiment 4 – Evaluating the RAG system

Each test question has keywords that a correct answer must contain. Two things are measured separately, so a failure can be traced to its cause:

* **Retrieval hit@k** – does any of the top-k retrieved chunks contain the keyword?
* **Answer accuracy** – does the final answer contain the keyword?

If retrieval misses, improve chunking or search; if retrieval hits but the answer is wrong, improve the prompt or model.

In [29]:
eval_set = [
    ("How long can the Wren S2 run in Eco mode?",                     ["180"]),
    ("What does error code E4 mean?",                                 ["main brush"]),
    ("How many days do I have to return the robot for a full refund?", ["30 days"]),
    ("How much does Care+ Premium cost per year?",                     ["99"]),
    ("Which Wi-Fi band does the Wren S2 support?",                     ["2.4"]),
    ("How often should I replace the HEPA filter?",                    ["3 months", "three months"]),
    ("How high a door threshold can the robot climb over?",            ["2.0 cm", "2 cm"]),
    ("How much is a replacement battery?",                             ["79"]),
    ("How long is the battery warranty?",                              ["18-month", "18 months", "eighteen"]),
    ("What is the customer support phone number?",                     ["555-0142"]),
]


def contains_any(text, keywords):
    text = text.lower()
    return any(k.lower() in text for k in keywords)


eval_rows = []
for question, keywords in eval_set:
    matches = search_kb(question)
    hit_rank = next((r for r, m in enumerate(matches, start=1) if contains_any(m.metadata["text"], keywords)), None)
    answer, _ = rag_answer(question)
    eval_rows.append({
        "question": question,
        "top-1 score": round(matches[0].score, 3),
        "hit rank": hit_rank if hit_rank else "-",
        f"retrieval hit@{TOP_K}": "✓" if hit_rank else "✗",
        "answer correct": "✓" if contains_any(answer, keywords) else "✗",
        "answer": answer,
    })

eval_df = pd.DataFrame(eval_rows)
pd.set_option("display.max_colwidth", 120)
display(eval_df)

hit_rate = (eval_df[f"retrieval hit@{TOP_K}"] == "✓").mean()
accuracy = (eval_df["answer correct"] == "✓").mean()
print(f"Retrieval hit@{TOP_K}: {hit_rate:.0%}")
print(f"Answer accuracy:   {accuracy:.0%}")

,question,top-1 score,hit rank,retrieval hit@4,answer correct,answer
0,How long can the Wren S2 run in Eco mode?,0.614,1,✓,✓,The Wren S2 can run for up to 180 minutes in Eco mode.
1,What does error code E4 mean?,0.519,1,✓,✓,"Error code E4 means ""Main brush tangled."" To resolve this issue, follow these steps:\n\n1. Remove the main brush.\n2..."
2,How many days do I have to return the robot for a full refund?,0.475,2,✓,✓,You can return the robot within 30 days of delivery for a full refund. If the purchase was delivered between Novembe...
3,How much does Care+ Premium cost per year?,0.479,1,✓,✓,Care+ Premium costs $99 per year.
4,Which Wi-Fi band does the Wren S2 support?,0.541,2,✓,✓,The Wren S2 supports the 2.4 GHz Wi-Fi band (802.11 b/g/n). 5 GHz networks are not supported.
5,How often should I replace the HEPA filter?,0.503,1,✓,✓,You should replace the HEPA filter every 3 months.
6,How high a door threshold can the robot climb over?,0.444,4,✓,✓,The Wren S2 can climb over door thresholds up to 2.0 cm high.
7,How much is a replacement battery?,0.341,3,✓,✓,A replacement battery costs $79.00.
8,How long is the battery warranty?,0.591,1,✓,✓,The battery has an 18-month warranty.
9,What is the customer support phone number?,0.420,3,✓,✓,The customer support phone number for Larkspur is 1-800-555-0142 (United States and Canada).


Retrieval hit@4: 100%
Answer accuracy:   100%


# 11. Chat with WrenHelper

Like the class demo's "run selected cell and all below" loop, but in one cell: type a question, press Enter, and repeat. Type `exit` (or press Enter on an empty line) to stop. The conversation memory from section 6 carries over.

Ideas to try: *"How do I set up the robot for the first time?"*, *"My mopping leaves streaks"*, *"Do you ship to Canada and how long does it take?"*, *"Is my data private?"*

In [ ]:
RUN_INTERACTIVE_CHAT = True

if RUN_INTERACTIVE_CHAT:
    print("Ask WrenHelper anything about the Wren S2. Type 'exit' to stop.\n")
    while True:
        user_q = input("User Input: ").strip()
        if user_q.lower() in {"", "exit", "quit", "q"}:
            print("Chat ended.")
            break
        print("\nWrenHelper:")
        print(wrap(rag_chat(user_q)))
        print()

Ask WrenHelper anything about the Wren S2. Type 'exit' to stop.

User Input: what is the battery life?

WrenHelper:
    The Wren S2 has a runtime of up to 180 minutes in Eco mode, 120 minutes in Standard mode, and 75 minutes
    in Max mode.

User Input: what is the cost?

WrenHelper:
    The suggested retail price of the Wren S2, which includes the robot and dock, is $649 USD.

User Input: any discount?

WrenHelper:
    Yes, you can receive 10% off accessories for the Wren S2.



# 12. Conclusions

**What was built.** A complete RAG pipeline: the knowledge-base PDF is extracted and cleaned with PyMuPDF, split into overlapping 6-line chunks, embedded with `text-embedding-3-small` (1536 dimensions), and stored in a Pinecone serverless index under its own namespace. At question time, the question is embedded, the top-4 chunks are retrieved by cosine similarity, wrapped in `####`, and passed with the system prompt and chat memory to `gpt-4o-mini`.

**What the experiments show.**

* **Without vs. with RAG (section 7).** The facts in this knowledge base exist only in the PDF, so the model on its own can only decline or guess. With retrieval, the same model answers with the exact error-code meanings, warranty periods and prices from the document. This is the core value of RAG: new or private knowledge without retraining.
* **Fact-checking (section 8).** Retrieved evidence turns the model into a verifier with three outcomes. The *Unknown* class matters: a claim that the document never mentions should not be marked False. Logprobs give a confidence score for each verdict, which could be used to send low-confidence cases to a human.
* **Guardrails (section 9).** The delimiter-based system prompt keeps the assistant inside the knowledge base and makes it say "I'm not certain…" instead of inventing facts.
* **Evaluation (section 10).** Measuring retrieval hit rate separately from answer accuracy shows where an error comes from: the search step or the generation step.

**Limitations and next steps.**

* Fixed-size line chunks can cut a table row or section in half; chunking by headings would keep each topic together.
* Exact identifiers such as "E9" or part numbers are a weak spot for pure dense search; hybrid search (keyword/BM25 + embeddings) or a reranker would help.
* Keyword matching is a crude correctness test; an LLM-as-a-judge evaluation (Week 7) would grade answers more fairly.
* Answers could cite the chunk IDs they used, so users can verify them.

# 13. Optional cleanup

Uncomment to delete this homework's vectors or the whole index. **Please be careful:** deleting an index cannot be undone.

In [ ]:
# index.delete(delete_all=True, namespace=NAMESPACE)   # delete only this knowledge base's vectors
# pc.delete_index(index_name)                            # delete the whole index